In [2]:
# ============================================
# MARIO RL
# Stage 1 + Stage 2 + Stage 3 + Replay Memory
# ============================================
# from Metric_Logger import MetricLogger
import torch
import random
import numpy as np
import itertools
from PIL import Image
import os

from gym import Wrapper
from gym.spaces import Box

import gym_super_mario_bros
from nes_py.wrappers import JoypadSpace


# ============================================
# Import Replay Memory Notebook
# ============================================

import import_ipynb
from Replay_Memory import ReplayMemory
from model import MarioNet

# ============================================
# Skip 4 frames
# ============================================

class SkipFrame(Wrapper):

    def __init__(self, env, skip=4):
        super().__init__(env)
        self.skip = skip

    def step(self, action):

        total_reward = 0

        for _ in range(self.skip):

            state, reward, terminated, truncated, info = self.env.step(action)

            total_reward += reward

            if terminated or truncated:
                break

        return state, total_reward, terminated, truncated, info


# ============================================
# Grayscale
# ============================================

class GrayScale(Wrapper):

    def __init__(self, env):
        super().__init__(env)

        self.observation_space = Box(
            low=0,
            high=255,
            shape=(240, 256, 1),
            dtype=np.uint8
        )

    def reset(self):

        state, info = self.env.reset()

        state = np.mean(
            state,
            axis=2
        ).astype(np.uint8)

        state = state[:, :, None]

        return state, info

    def step(self, action):

        state, reward, terminated, truncated, info = self.env.step(action)

        state = np.mean(
            state,
            axis=2
        ).astype(np.uint8)

        state = state[:, :, None]

        return state, reward, terminated, truncated, info


# ============================================
# Resize to 84 × 84
# ============================================

class Resize(Wrapper):

    def __init__(self, env):
        super().__init__(env)

        self.observation_space = Box(
            low=0,
            high=255,
            shape=(84, 84, 1),
            dtype=np.uint8
        )

    def resize(self, state):

        state = Image.fromarray(
            state.squeeze()
        )

        state = state.resize(
            (84, 84)
        )

        return np.array(state)[:, :, None]

    def reset(self):

        state, info = self.env.reset()

        return self.resize(state), info

    def step(self, action):

        state, reward, terminated, truncated, info = self.env.step(action)

        return (
            self.resize(state),
            reward,
            terminated,
            truncated,
            info
        )


# ============================================
# Stack 4 frames
# ============================================

class FrameStack(Wrapper):

    def __init__(self, env, num_frames=4):
        super().__init__(env)

        self.frames = []

        self.num_frames = num_frames

        self.observation_space = Box(
            low=0,
            high=255,
            shape=(num_frames, 84, 84),
            dtype=np.uint8
        )

    def reset(self):

        state, info = self.env.reset()

        self.frames = [state] * self.num_frames

        return (
            np.array(self.frames).squeeze(-1),
            info
        )

    def step(self, action):

        state, reward, terminated, truncated, info = self.env.step(action)

        self.frames.append(state)

        self.frames = self.frames[-self.num_frames:]

        return (
            np.array(self.frames).squeeze(-1),
            reward,
            terminated,
            truncated,
            info
        )



Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:

# ============================================
# Mario Agent
# ============================================

class Mario:

    def __init__(
        self,
        state_dim,
        action_dim,
        device
    ):

        self.state_dim = state_dim
        self.action_dim = action_dim
        self.device = device
                
        self.gamma = 0.9
        
        self.batch_size = 32
        
        self.burnin = 1000
        
        self.learn_every = 3
        
        self.sync_every = 1000
        
        self.model_path="mario_best.pt"
        self.mario_checkpoint="mario_checkpoint.pt"
        self.step = 0

        # Exploration
        self.exploration_rate = 1.0
        self.exploration_rate_decay = 0.9999975
        self.exploration_rate_min = 0.1

        # Replay Memory
        self.memory = ReplayMemory(
            capacity=100000
        )
        self.best_reward = float("-inf")
        self.loss_fn = torch.nn.SmoothL1Loss()

        # Neural Networks
                     
        self.online = MarioNet(
            state_dim,
            action_dim
        ).to(self.device)
        
        self.target = MarioNet(
            state_dim,
            action_dim
        ).to(self.device)
        
        self.optimizer = torch.optim.Adam(
                self.online.parameters(),
                lr=0.00025
            )
    
        
        if os.path.exists(self.mario_checkpoint):

            checkpoint = torch.load(
                self.mario_checkpoint,
                map_location=self.device
            )
        
            self.online.load_state_dict(
                checkpoint["online"]
            )
        
            self.target.load_state_dict(
                checkpoint["target"]
            )
        
            self.optimizer.load_state_dict(
                checkpoint["optimizer"]
            )
        
            self.exploration_rate = checkpoint["epsilon"]
            
            self.best_reward = checkpoint.get(
                "best_reward",
                float("-inf"))        
            self.step = checkpoint["step"]
        
        else:
        
            self.target.load_state_dict(
            self.online.state_dict()
        )
        
        self.target.eval()


    # ========================================
    # Choose Action
    # ========================================

    def act(self, state,is_tr):

        if is_tr and random.random() < self.exploration_rate:
            return random.randrange(
                self.action_dim
            )
         
        state = torch.tensor(
            state,
            dtype=torch.float32
        ).unsqueeze(0).to(self.device)

        q_values = self.online(state)

        return q_values.argmax().item()
        
    def cache(
    self,
    state,
    next_state,
    action,
    reward,
    done):
      self.memory.push(
        state,
        action,
        reward,
        next_state,
        done
    )

    def learn(self):
    
        self.step += 1
    
        # Not enough experiences yet
        if len(self.memory) < self.burnin:
            return
    
        # Learn only every few steps
        if self.step % self.learn_every != 0:
            return
    
        # Train online network
        loss = self._learn(self.batch_size)
    
        # Sync target network
        if self.step % self.sync_every == 0:
            self.target.load_state_dict(
                self.online.state_dict()
            )
            torch.save({
                "online": self.online.state_dict(),
                "target": self.target.state_dict(),
                "optimizer": self.optimizer.state_dict(),
                "epsilon": self.exploration_rate,
                "step": self.step,
                "best_reward": self.best_reward
            }, "mario_checkpoint.pt")
    
        return loss

    def _learn(self, batch_size):
    
        # Get experiences
        batch = self.memory.sample(batch_size)
    
        states, actions, rewards, next_states, dones = zip(*batch)
    
    
        # Convert to tensors
        states = torch.from_numpy(
            np.array(states)
        ).float().to(self.device)
        
        actions = torch.tensor(
            actions,
            dtype=torch.long
        ).to(self.device)
        
        rewards = torch.tensor(
            rewards,
            dtype=torch.float32
        ).to(self.device)
        
        next_states = torch.from_numpy(
            np.array(next_states)
        ).float().to(self.device)
        
        dones = torch.tensor(
            dones,
            dtype=torch.float32
        ).to(self.device)
    
    
        # ====================================
        # TD Estimate
        # ====================================
    
        current_q = self.online(states)
    
        current_q = current_q.gather(
            1,
            actions.unsqueeze(1)
        ).squeeze(1)
    
    
        # ====================================
        # Double DQN Target
        # ====================================
    
        with torch.no_grad():
    
            # Online network chooses action
            next_actions = self.online(
                next_states
            ).argmax(
                dim=1,
                keepdim=True
            )
    
            # Target network evaluates action
            next_q = self.target(
                next_states
            ).gather(
                1,
                next_actions
            ).squeeze(1)
    
            # TD Target
            target_q = rewards + (
                1 - dones
            ) * 0.9 * next_q
    
    
        # ====================================
        # Calculate Loss
        # ====================================
    
        loss = self.loss_fn(
            current_q,
            target_q
        )
    
    
        # ====================================
        # Update Online Network
        # ====================================
    
        self.optimizer.zero_grad()
    
        loss.backward()
    
        self.optimizer.step()
    
    
        return loss.item()
        
    def run(self, is_training=True):

        # ============================================
        # Load best model for evaluation
        # ============================================
    
        if not is_training:
    
            self.online.load_state_dict(
                torch.load(
                    self.model_path,
                    map_location=self.device
                )
            )
    
            self.online.eval()
    
        # ============================================
        # Training setup
        # ============================================
    
        if is_training:
    
            self.online.train()
    
            self.target.load_state_dict(
                self.online.state_dict()
            )
    
            self.target.eval()
    
 
    
        # ============================================
        # Episodes
        # ============================================
    
        max_episodes = 100 if not is_training else 100000
    
        for episode in range(max_episodes):
    
            state, info = env.reset()
    
            done = False
            episode_reward = 0
    
            while not done:
    
                action = self.act(
                    state,
                    is_training
                )
    
                next_state, reward, terminated, truncated, info = env.step(action)
    
                if not is_training:
                    env.render()
    
                done = terminated or truncated
    
                if is_training:
    
                    self.cache(
                        state,
                        next_state,
                        action,
                        reward,
                        done
                    )
    
                    self.learn()
    
                    self.exploration_rate = max(
                        self.exploration_rate *
                        self.exploration_rate_decay,
                        self.exploration_rate_min
                    )
    
                state = next_state
    
                episode_reward += reward
    
            # ========================================
            # Save best model
            # ========================================
    
            if is_training and episode_reward > self.best_reward:
    
                self.best_reward = episode_reward
    
                torch.save(
                    self.online.state_dict(),
                    self.model_path
                )
    
            print(
                f"Episode: {episode + 1} | "
                f"Reward: {episode_reward} | "
                f"Epsilon: {self.exploration_rate:.4f}"
            )

In [ ]:
if __name__ == "__main__":
    is_training=True
    env = gym_super_mario_bros.make(
     "SuperMarioBros-1-1-v0",
     render_mode="human" if not is_training else None
)

    env = JoypadSpace( env,[
            ["right"],          # 0
            ["right", "A"],     # 1
            ["right", "B"]      # 2
        ]
    )
 

    env = SkipFrame(env)
    env = GrayScale(env)
    env = Resize(env)
    env = FrameStack(env)
 

    state_dim = env.observation_space.shape
    action_dim = env.action_space.n
 

    device = "cuda" if torch.cuda.is_available() else "cpu"
 

    mario = Mario(
        state_dim,
        action_dim,
        device
    )
 

    mario.run(
        is_training
    )


 

    env.close()

Episode: 1 | Reward: 617.0 | Epsilon: 0.9276
Episode: 2 | Reward: 638.0 | Epsilon: 0.9272
Episode: 3 | Reward: 711.0 | Epsilon: 0.9257
